# Theory

Data leakage happens when information that would not be genuinely available at prediction time — or information from outside the training set — sneaks into the model training process. The result is a model that looks excellent during development (high training/validation scores) but performs much worse (or fails outright) in real deployment, because it secretly relied on information it won't actually have access to when making new predictions.

Leakage is one of the most common — and most dangerous, because it's often invisible — mistakes in applied machine learning.

| **Type**                     | **Description**                                                                                                                                                             |
| ---------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Target Leakage**           | A feature contains information that is a direct or indirect proxy for the target and would **not be known before the outcome occurs**.                                      |
| **Train-Test Contamination** | Information from the **test/validation set influences the training process** — for example, fitting a scaler, imputer, or encoder on the **full dataset before splitting**. |


**Target Leakage Risks**

Reviewing used_cars_dataset.csv's columns, most are legitimately available at listing time. But a few require careful thought:

Year and Age_Years together — this is technically multicollinearity (Lecture 3 of the Feature Selection series) rather than leakage, but it illustrates a related idea: redundant encodings of the same fact can distort a model even without being "wrong" information.
If the dataset ever included a column like Final_Sale_Price_After_Negotiation or Days_Listed_Before_Sale (not present here, but common in real used-car marketplaces) — these would be filled in only after the sale happens, making them invalid features for predicting the initial listing price. Always ask: "would I know this value at the moment I need to make the prediction?"
City is safe — it's known at listing time and can legitimately reflect real regional price differences.

**Train-Test Contamination Risks (the More Common Trap)**

This is where most practical leakage bugs happen, and this dataset offers concrete examples:

❌ Leaky pattern — scaling before splitting:

In [ ]:
from sklearn.preprocessing import StandardScaler

# WRONG: scaler sees test set statistics before the split even happens
scaler = StandardScaler()
df[["Kilometer", "Engine_CC", "Power_BHP"]] = scaler.fit_transform(
    df[["Kilometer", "Engine_CC", "Power_BHP"]]
)
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["Price"]), df["Price"])

**Correct pattern — split first, fit only on train:**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Price"]), df["Price"], test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train[["Kilometer", "Engine_CC", "Power_BHP"]])          # fit on TRAIN only

X_train_scaled = scaler.transform(X_train[["Kilometer", "Engine_CC", "Power_BHP"]])
X_test_scaled = scaler.transform(X_test[["Kilometer", "Engine_CC", "Power_BHP"]])  # transform only

**Leakage via Missing Value Imputation**

If Mileage_kmpl had missing values and we imputed them using the overall dataset mean before splitting, the test set's missing values would be filled using information that includes the test set itself — a subtle form of contamination.

**Practical Code — Using a Pipeline to Prevent Leakage Automatically**

The safest defense against leakage isn't remembering to do things "in the right order" manually — it's using scikit-learn's Pipeline, which automatically ensures every preprocessing step is fit only on training folds during cross-validation, and refit correctly for the final test evaluation.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("used_cars_dataset.csv").drop(columns=["Car_ID"])
X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["Age_Years", "Kilometer", "Mileage_kmpl", "Engine_CC", "Power_BHP", "Seats"]
categorical_features = ["Brand", "Model", "Fuel_Type", "Transmission", "Owner_Type", "Color", "City"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),   # fit only within each CV fold / on train
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

# cross_val_score automatically re-fits the ENTIRE pipeline (imputer, scaler, encoder, model)
# fresh within each fold — no leakage across folds
cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2")
print("Leak-safe cross-validated R²:", cv_scores.mean())

full_pipeline.fit(X_train, y_train)
print("Final test R²:", full_pipeline.score(X_test, y_test))